In [2]:
%pip install monai[all] nibabel matplotlib torch tqdm

     ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
     ---------- ----------------------------- 1.0/4.1 MB 20.1 MB/s eta 0:00:01
     ---------- ----------------------------- 1.0/4.1 MB 20.1 MB/s eta 0:00:01
     ---------- ----------------------------- 1.0/4.1 MB 20.1 MB/s eta 0:00:01
     ---------- ----------------------------- 1.0/4.1 MB 20.1 MB/s eta 0:00:01
     ---------- ----------------------------- 1.0/4.1 MB 20.1 MB/s eta 0:00:01
     ---------- ----------------------------- 1.0/4.1 MB 20.1 MB/s eta 0:00:01
     -------------------- ------------------- 2.1/4.1 MB 1.5 MB/s eta 0:00:02
     ------------------------------ --------- 3.1/4.1 MB 1.8 MB/s eta 0:00:01
     ------------------------------ --------- 3.1/4.1 MB 1.8 MB/s eta 0:00:01
     ---------------------------------------- 4.1/4.1 MB 2.0 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: start

  error: subprocess-exited-with-error
  
  × Building wheel for pyamg (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [161 lines of output]
      C:\Users\whate\AppData\Local\Temp\pip-build-env-1np2jcgj\overlay\Lib\site-packages\setuptools_scm\_integration\deprecation.py:9: UserWarning: setup.cfg: at [metadata]
      version = attr: ... is forcing setuptools to override the version setuptools-scm did already set
      When using setuptools-scm it's invalid to use setuptools dynamic version as well, please remove it.
      Setuptools-scm is responsible for setting the version, forcing setuptools to override creates errors.
        warnings.warn(
      C:\Users\whate\AppData\Local\Temp\pip-build-env-1np2jcgj\overlay\Lib\site-packages\vcs_versioning\_fallback_workdir.py:248: UserWarning: git archive did not support describe output
        return archival_to_version(data, config)
      C:\Users\whate\AppData\Local\Temp\pip-build-env-1np2jcgj\overlay\Lib\site-packages\vcs_

In [3]:
# Run this cell to configure the environment
import os
import glob
import json
import torch
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    Spacingd, ScaleIntensityRanged, CropForegroundd, 
    RandCropByPosNegLabeld, EnsureTyped
)
from monai.data import Dataset, DataLoader, select_cross_validation_folds
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference


ModuleNotFoundError: No module named 'monai'

In [ ]:
data_dir = "C:/Users/whate/Documents/Brain CT Hemorrage Detection/Dataset/labeled" # Update this to your local path

# Map paths using structural JSON or localized directory paths
images = sorted(glob.glob(os.path.join(data_dir, "ground_truths", "*.nii.gz")))
labels = sorted(glob.glob(os.path.join(data_dir, "images", "*.nii.gz")))

data_dicts = [{"image": img, "label": lbl} for img, lbl in zip(images, labels)]

# Standard split rule: 80% train, 20% validation
train_size = int(0.8 * len(data_dicts))
train_files, val_files = data_dicts[:train_size], data_dicts[train_size:]
print(f"Train samples: {len(train_files)} | Validation samples: {len(val_files)}")


In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    # Standardizing spatial resolution across volumes
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 5.0), mode=("bilinear", "nearest")),
    # Crucial step: Windowing to focus purely on ICH features (Soft Tissue/Blood Window)
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=100, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    # Extracting smaller cubes to avoid OOM
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 32), 
        pos=1, neg=1, num_samples=4, # Extract 4 patches per volume dynamically
        image_key="image",
        image_threshold=0
    ),
    EnsureTyped(keys=["image", "label"]),
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 5.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=100, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    EnsureTyped(keys=["image", "label"]),
])

# Generate Dataloaders
train_ds = Dataset(data=train_files, transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4)

val_ds = Dataset(data=val_files, transform=val_transforms)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=2)
 

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 6 Output channels: 1 Background + 5 ICH Subtypes (EDH, IPH, IVH, SAH, SDH)
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=6,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH,
).to(device)

loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
dice_metric = DiceMetric(include_background=False, reduction="mean")


In [ ]:
max_epochs = 100
val_interval = 2
best_metric = -1
best_metric_epoch = -1

for epoch in range(max_epochs):
    model.train()
    epoch_loss = 0
    with tqdm(train_loader, unit="batch") as tepoch:
        tepoch.set_description(f"Epoch {epoch + 1}/{max_epochs}")
        for batch_data in tepoch:
            inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())

    # Evaluation Cycle
    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                roi_size = (96, 96, 32)
                # Stitching predictions together seamlessly
                val_outputs = sliding_window_inference(val_inputs, roi_size, 4, model)
                dice_metric(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            print(f"Epoch {epoch+1} - Validation Average Dice Score: {metric:.4f}")

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), "best_bhsd_unet_model.pth")
                print("New optimal weights saved successfully.")
